# Matrix-Level φ Redundancy Optimization

This notebook tests whether the algebraic redundancy of base-φ representations can be exploited **globally across a neural-network weight matrix**.

The central identity is

\[
100_\phi = 011_\phi
\]

and its shifted forms.

Instead of changing the quantized numerical weights, we keep every selected φ codebook value fixed and change only its equivalent representation.

We test:

1. Per-weight minimum-popcount representations.
2. Weighted plane penalties.
3. Exhaustive search over all \(2^8=256\) subsets of φ digit planes.
4. Whether the same quantized matrix can be represented with fewer active planes.
5. The tradeoff between number of active planes and total active digits.

This isolates the benefit of **φ redundancy itself**: numerical weights, quantization MSE, and model accuracy must remain unchanged when only equivalent representations are rewritten.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt
import math
import itertools
from collections import defaultdict

torch.manual_seed(42)
np.random.seed(42)

print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)


## 1. Load MNIST


In [ ]:
transform = transforms.ToTensor()

train_dataset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=128, shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=128, shuffle=False
)

print("Train:", len(train_dataset))
print("Test:", len(test_dataset))


## 2. Train the same small MLP


In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu(x)
        return self.fc2(x)

model = SimpleMLP()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(3):
    model.train()
    total_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/3, "
        f"Loss: {total_loss / len(train_loader):.4f}"
    )


In [ ]:
def evaluate(model):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            pred = outputs.argmax(dim=1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()

    return correct / total

fp32_accuracy = evaluate(model)

W = model.fc1.weight.detach().clone()
W_original = W.clone()

print(f"FP32 accuracy: {fp32_accuracy*100:.2f}%")
print("Weight matrix:", tuple(W.shape))
print("Weights:", W.numel())


## 3. Build the complete φ representation space

For \(K=8\),

\[
c(d)=\sum_{k=1}^{8}d_k\phi^{-k},\qquad d_k\in\{0,1\}.
\]

We group bit patterns by numerical value. Each group contains all equivalent representations available within the eight-digit window.


In [ ]:
phi = (1 + math.sqrt(5)) / 2
K = 8

phi_powers = np.array(
    [phi ** (-k) for k in range(1, K + 1)],
    dtype=np.float64
)

groups = defaultdict(list)

for bits in itertools.product([0, 1], repeat=K):
    value = float(np.dot(np.array(bits), phi_powers))
    groups[round(value, 10)].append(tuple(bits))

print("Raw patterns:", 2**K)
print("Unique numerical magnitudes:", len(groups))
print("Redundant patterns:", 2**K - len(groups))

multiplicities = [len(v) for v in groups.values()]
print("Mean representations/value:", np.mean(multiplicities))
print("Maximum representations/value:", np.max(multiplicities))


## 4. Canonical per-weight representation

This reproduces the previous strategy: for each numerical value, choose the equivalent representation with the fewest `1`s.

This is a **local** optimization.


In [ ]:
canonical_phi = {}

for value, representations in groups.items():
    canonical_phi[value] = min(
        representations,
        key=lambda bits: (sum(bits), bits)
    )

canonical_counts = np.array(
    [sum(bits) for bits in canonical_phi.values()]
)

print("Average canonical active digits per codebook value:",
      canonical_counts.mean())


## 5. Quantize the trained matrix once

This step chooses the nearest φ **numerical value** for each trained weight.

After this cell, those numerical choices are frozen. Every later experiment is allowed to change only the equivalent digit representation, not the quantized weight.


In [ ]:
phi_keys = list(groups.keys())
phi_values = np.array(phi_keys, dtype=np.float64)

max_phi_value = phi_values.max()
phi_scale = W.abs().max().item() / max_phi_value

flat_W = W.flatten().cpu().numpy()

selected_phi_keys = []
W_phi_flat = np.zeros_like(flat_W, dtype=np.float32)

for i, original in enumerate(flat_W):
    target = abs(float(original)) / phi_scale

    idx = int(np.argmin(np.abs(phi_values - target)))
    key = phi_keys[idx]

    selected_phi_keys.append(key)

    sign = -1.0 if original < 0 else 1.0
    W_phi_flat[i] = sign * phi_scale * key

W_phi = torch.tensor(
    W_phi_flat.reshape(W.shape),
    dtype=W.dtype
)

phi_mse = torch.mean((W - W_phi) ** 2).item()
phi_mae = torch.mean(torch.abs(W - W_phi)).item()

with torch.no_grad():
    model.fc1.weight.copy_(W_phi)

phi_accuracy = evaluate(model)

with torch.no_grad():
    model.fc1.weight.copy_(W_original)

print("φ scale:", phi_scale)
print("φ MSE:", phi_mse)
print("φ MAE:", phi_mae)
print(f"φ accuracy: {phi_accuracy*100:.2f}%")


## 6. Build the locally sparsest matrix representation


In [ ]:
local_bits = np.array(
    [canonical_phi[key] for key in selected_phi_keys],
    dtype=np.int8
)

print("Shape:", local_bits.shape)
print("Total active digits:", local_bits.sum())
print("Average active digits/weight:", local_bits.sum(axis=1).mean())
print("Overall digit density:", local_bits.mean())
print("Overall digit sparsity:", 1 - local_bits.mean())

local_plane_counts = local_bits.sum(axis=0)

print("\nPer-plane active digits:")
for k, count in enumerate(local_plane_counts, start=1):
    print(f"B{k}: {int(count)}")


## 7. Visualize the eight binary φ planes


In [ ]:
B_local = [
    local_bits[:, k].reshape(W.shape)
    for k in range(K)
]

fig, axes = plt.subplots(2, 4, figsize=(16, 6))

for k, ax in enumerate(axes.flat):
    ax.imshow(B_local[k], aspect="auto", cmap="binary")
    ax.set_title(
        f"B{k+1}: φ^-{k+1}\n"
        f"ones={int(B_local[k].sum())}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()


## 8. Weighted plane optimization

For an equivalent representation \(r=(d_1,\ldots,d_8)\), define

\[
C(r)=\sum_k c_kd_k.
\]

With all \(c_k=1\), this is ordinary popcount.

If one plane is assigned a large cost, the optimizer will prefer equivalent representations that avoid that plane when possible.


In [ ]:
def choose_representation(representations, plane_costs):
    return min(
        representations,
        key=lambda bits: (
            sum(bit * cost for bit, cost in zip(bits, plane_costs)),
            sum(bits),
            bits
        )
    )

def optimize_with_plane_costs(selected_keys, plane_costs):
    return np.array(
        [
            choose_representation(groups[key], plane_costs)
            for key in selected_keys
        ],
        dtype=np.int8
    )

# Penalize the final plane heavily.
plane_costs = np.ones(K)
plane_costs[-1] = 10.0

weighted_bits = optimize_with_plane_costs(
    selected_phi_keys,
    plane_costs
)

print("Local total ones:", local_bits.sum())
print("Weighted total ones:", weighted_bits.sum())

print("\nPlane counts before -> after:")
for k in range(K):
    print(
        f"B{k+1}: "
        f"{int(local_bits[:, k].sum())} -> "
        f"{int(weighted_bits[:, k].sum())}"
    )


## 9. Verify that representation rewriting changed no numerical values

This is essential. The matrix-level representation may change, but the represented φ codebook magnitude for every weight must remain identical.


In [ ]:
def decode_bits(bits_matrix):
    return bits_matrix.astype(np.float64) @ phi_powers

decoded_local = decode_bits(local_bits)
decoded_weighted = decode_bits(weighted_bits)

selected_values = np.array(
    selected_phi_keys,
    dtype=np.float64
)

local_error = np.max(np.abs(decoded_local - selected_values))
weighted_error = np.max(np.abs(decoded_weighted - selected_values))

print("Maximum local decode error:", local_error)
print("Maximum weighted decode error:", weighted_error)

assert np.allclose(
    decoded_local,
    selected_values,
    atol=1e-8
)

assert np.allclose(
    decoded_weighted,
    selected_values,
    atol=1e-8
)

print("PASS: all quantized numerical values are unchanged.")


# 10. Global plane-subset optimization

Now we ask a stronger question.

For every subset

\[
S\subseteq\{B_1,\ldots,B_8\},
\]

can **every selected φ value in the entire matrix** be represented using only planes in \(S\)?

There are only

\[
2^8=256
\]

subsets, so we can test all of them exactly.

For every feasible subset we also choose, for each weight, the minimum-popcount equivalent representation that uses only the allowed planes.


In [ ]:
# Only distinct selected numerical values matter for feasibility.
used_keys = sorted(set(selected_phi_keys))

print("Distinct φ magnitudes actually used by W:", len(used_keys))

def representation_uses_only(bits, allowed_mask):
    return all(
        (bit == 0) or allowed
        for bit, allowed in zip(bits, allowed_mask)
    )

def best_rep_for_key_under_mask(key, allowed_mask):
    candidates = [
        bits
        for bits in groups[key]
        if representation_uses_only(bits, allowed_mask)
    ]

    if not candidates:
        return None

    return min(
        candidates,
        key=lambda bits: (sum(bits), bits)
    )


In [ ]:
subset_results = []

for mask_int in range(1, 2**K):

    allowed_mask = tuple(
        bool(mask_int & (1 << k))
        for k in range(K)
    )

    best_by_key = {}
    feasible = True

    for key in used_keys:
        best = best_rep_for_key_under_mask(
            key,
            allowed_mask
        )

        if best is None:
            feasible = False
            break

        best_by_key[key] = best

    if not feasible:
        continue

    bits_matrix = np.array(
        [best_by_key[key] for key in selected_phi_keys],
        dtype=np.int8
    )

    active_planes = int(
        np.count_nonzero(bits_matrix.sum(axis=0))
    )

    total_ones = int(bits_matrix.sum())

    subset_results.append({
        "mask_int": mask_int,
        "allowed_mask": allowed_mask,
        "allowed_planes": sum(allowed_mask),
        "active_planes": active_planes,
        "total_ones": total_ones,
        "avg_ones": total_ones / len(selected_phi_keys),
        "bits_matrix": bits_matrix
    })

print("Feasible subsets:", len(subset_results))


## 11. Find the minimum number of planes

Among all exact representations of the same quantized matrix, find the solution using the fewest active φ planes.


In [ ]:
min_planes = min(
    r["active_planes"]
    for r in subset_results
)

fewest_plane_solutions = [
    r for r in subset_results
    if r["active_planes"] == min_planes
]

best_fewest_plane = min(
    fewest_plane_solutions,
    key=lambda r: r["total_ones"]
)

print("Original active planes:",
      np.count_nonzero(local_bits.sum(axis=0)))

print("Minimum achievable active planes:",
      min_planes)

print("Best total ones at that plane count:",
      best_fewest_plane["total_ones"])

print("Average ones/weight:",
      best_fewest_plane["avg_ones"])

print("\nActive plane mask:")
for k, allowed in enumerate(
    best_fewest_plane["allowed_mask"],
    start=1
):
    print(f"B{k}: {'ON' if allowed else 'OFF'}")


## 12. Verify the globally optimized representation exactly

Again, the represented quantized values must not change.


In [ ]:
global_bits = best_fewest_plane["bits_matrix"]

decoded_global = decode_bits(global_bits)

max_global_error = np.max(
    np.abs(decoded_global - selected_values)
)

print("Maximum decode error:", max_global_error)

assert np.allclose(
    decoded_global,
    selected_values,
    atol=1e-8
)

print("PASS: global plane optimization preserves every quantized value.")

print()
print("Therefore:")
print("φ MSE remains:", phi_mse)
print(f"φ model accuracy remains: {phi_accuracy*100:.2f}%")


## 13. Compare plane activity before and after


In [ ]:
local_counts = local_bits.sum(axis=0)
global_counts = global_bits.sum(axis=0)

x = np.arange(1, K + 1)
width = 0.35

plt.figure(figsize=(10, 5))

plt.bar(
    x - width/2,
    local_counts,
    width,
    label="Local min-popcount"
)

plt.bar(
    x + width/2,
    global_counts,
    width,
    label="Global min-plane solution"
)

plt.xlabel("φ digit plane")
plt.ylabel("Active digits")
plt.title("Plane Activity Before vs After Global Optimization")
plt.xticks(x)
plt.legend()
plt.show()


## 14. Find the Pareto frontier: active planes vs total active digits

Eliminating planes may force some weights to use more digits.

So there can be a tradeoff:

- fewer active planes;
- fewer total active digits.

We calculate the best feasible solution for every active-plane count.


In [ ]:
best_by_plane_count = {}

for r in subset_results:
    p = r["active_planes"]

    if (
        p not in best_by_plane_count
        or r["total_ones"] < best_by_plane_count[p]["total_ones"]
    ):
        best_by_plane_count[p] = r

print(
    f"{'Active planes':>13} "
    f"{'Total ones':>12} "
    f"{'Avg ones/weight':>17}"
)

for p in sorted(best_by_plane_count):
    r = best_by_plane_count[p]

    print(
        f"{p:>13} "
        f"{r['total_ones']:>12} "
        f"{r['avg_ones']:>17.6f}"
    )


In [ ]:
plane_counts = sorted(best_by_plane_count)

total_ones = [
    best_by_plane_count[p]["total_ones"]
    for p in plane_counts
]

plt.figure(figsize=(8, 5))

plt.plot(
    plane_counts,
    total_ones,
    marker="o"
)

plt.xlabel("Number of active φ planes")
plt.ylabel("Minimum total active digits")
plt.title("Exact φ Representation Tradeoff")
plt.grid(alpha=0.25)
plt.show()


## 15. Visualize globally optimized planes


In [ ]:
B_global = [
    global_bits[:, k].reshape(W.shape)
    for k in range(K)
]

fig, axes = plt.subplots(2, 4, figsize=(16, 6))

for k, ax in enumerate(axes.flat):
    ax.imshow(
        B_global[k],
        aspect="auto",
        cmap="binary"
    )

    ax.set_title(
        f"B{k+1}: φ^-{k+1}\n"
        f"ones={int(B_global[k].sum())}"
    )

    ax.axis("off")

plt.tight_layout()
plt.show()


# 16. Final experiment summary

This cell prints the quantities to send back for analysis.


In [ ]:
local_active_planes = int(
    np.count_nonzero(local_bits.sum(axis=0))
)

global_active_planes = int(
    np.count_nonzero(global_bits.sum(axis=0))
)

local_total = int(local_bits.sum())
global_total = int(global_bits.sum())

print("=" * 60)
print("MATRIX-LEVEL φ REDUNDANCY RESULTS")
print("=" * 60)

print()
print("Numerical quantization (held fixed)")
print("----------------------------------")
print("φ MSE:", phi_mse)
print("φ MAE:", phi_mae)
print(f"φ accuracy: {phi_accuracy*100:.2f}%")

print()
print("Local minimum-popcount representation")
print("-------------------------------------")
print("Active planes:", local_active_planes)
print("Total active digits:", local_total)
print(
    "Average active digits/weight:",
    local_total / W.numel()
)

print()
print("Global minimum-plane representation")
print("-----------------------------------")
print("Active planes:", global_active_planes)
print("Total active digits:", global_total)
print(
    "Average active digits/weight:",
    global_total / W.numel()
)

print()
print("Changes")
print("-------")
print(
    "Planes eliminated:",
    local_active_planes - global_active_planes
)

print(
    "Change in total active digits:",
    global_total - local_total
)

print(
    "Percent change in active digits:",
    100 * (global_total - local_total) / local_total
)

print()
print("Per-plane counts: local -> global")
for k in range(K):
    print(
        f"B{k+1}: "
        f"{int(local_counts[k])} -> "
        f"{int(global_counts[k])}"
    )

print()
print("Exact-value preservation:",
      np.allclose(decoded_global, selected_values, atol=1e-8))


## How to interpret the result

A particularly interesting outcome would be:

- fewer active planes,
- only a small increase (or even a decrease) in total active digits,
- exactly unchanged decoded φ values.

That would demonstrate that the redundancy

\[
100_\phi\leftrightarrow011_\phi
\]

provides a **global structural degree of freedom** after quantization.

It would still **not** prove lower runtime or energy on conventional hardware. The next stage would need a hardware-aware cost model or an implementation designed around digit planes.

If whole-plane elimination is weak, the next natural experiment is **block-level optimization**, where the matrix is divided into blocks and equivalent φ representations are selected to create structured zero blocks.
